Step 1 (Set Up and GPU check)

In [ ]:
import torch, torchvision, sys, platform
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('⚠️  No GPU detected. In Colab, go to Runtime → Change runtime type → select GPU.')

Python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
Torchvision: 0.23.0+cu126
CUDA available: True
GPU: Tesla T4


Step 2 (Install Extra Library)

In [ ]:
!pip -q install scikit-learn torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.0/983.0 kB 22.9 MB/s eta 0:00:00


Step 3 (import)

In [ ]:
import os, math, time, random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
from torchvision import datasets, transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
from torchmetrics.classification import MulticlassAccuracy

torch.backends.cudnn.benchmark = True
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

Step 4 (Connect Google Drive)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Step 5 (Configure Dataset Path)

In [ ]:
# If you downloaded the Kaggle dataset to Drive, point BASE_DIR to it.
# Example: '/content/drive/MyDrive/Kerry_AI(assignment)'
BASE_DIR = '/content/drive/MyDrive/Kerry_AI(assignment)'  # <-- change if needed

assert os.path.isdir(BASE_DIR), f"Base directory not found: {BASE_DIR}"

# Detect common structures:
# 1) BASE_DIR/Training/<class> ... and BASE_DIR/Testing/<class> ...
# 2) BASE_DIR/<class> ... (single folder with subfolders per class)
train_dir, val_dir, test_dir = None, None, None

cand_train = os.path.join(BASE_DIR, 'Training')
cand_test  = os.path.join(BASE_DIR, 'Testing')
cand_val   = os.path.join(BASE_DIR, 'Validation')

if os.path.isdir(cand_train) and os.path.isdir(cand_test):
    train_dir = cand_train
    test_dir  = cand_test
    if os.path.isdir(cand_val):
        val_dir = cand_val
    print('📦 Detected Training/Testing structure.')
else:
    # assume single root with class subfolders; we will split later
    train_dir = BASE_DIR
    print('📦 Detected single-folder dataset. Will create train/val/test splits.')

print('train_dir:', train_dir)
print('val_dir  :', val_dir)
print('test_dir :', test_dir)

📦 Detected Training/Testing structure.
train_dir: /content/drive/MyDrive/Kerry_AI(assignment)/Training
val_dir  : None
test_dir : /content/drive/MyDrive/Kerry_AI(assignment)/Testing


Step 6 (Select Folder for save the trained data set(ResNet-50 model))

In [ ]:
# Cell 7 - Select or create folder to save models
default_base = '/content/drive/MyDrive'
folder_name = input(f"Enter a folder name to save trained models (default: BrainTumor_Models): ") or "BrainTumor_Models"
save_dir = os.path.join(default_base, folder_name)
os.makedirs(save_dir, exist_ok=True)
print(f"✅ Models will be saved to: {save_dir}")

Enter a folder name to save trained models (default: BrainTumor_Models): ResNet-50 model
✅ Models will be saved to: /content/drive/MyDrive/ResNet-50 model


Step 7 (Define Transforms)

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

to_rgb = transforms.Lambda(lambda im: im.convert('RGB'))

train_tfms = transforms.Compose([
    to_rgb,
    transforms.RandomResizedCrop(224, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_tfms = transforms.Compose([
    to_rgb,
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

Step 8 (Build datasets and loaders)

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

if os.path.isdir(os.path.join(train_dir, 'glioma')) or os.path.isdir(os.path.join(train_dir, 'pituitary')):
    # Single-folder dataset with class subfolders
    full_ds = datasets.ImageFolder(train_dir, transform=None)  # transform set later per split
    num_classes = len(full_ds.classes)
    print('Classes:', full_ds.classes)

    # Split 80/10/10
    n = len(full_ds)
    n_train = int(0.8 * n)
    n_val   = int(0.1 * n)
    n_test  = n - n_train - n_val
    torch.manual_seed(42)
    train_idx, val_idx, test_idx = torch.utils.data.random_split(range(n), [n_train, n_val, n_test])
    train_ds = Subset(datasets.ImageFolder(train_dir, transform=train_tfms), train_idx.indices)
    val_ds   = Subset(datasets.ImageFolder(train_dir, transform=eval_tfms),   val_idx.indices)
    test_ds  = Subset(datasets.ImageFolder(train_dir, transform=eval_tfms),  test_idx.indices)
else:
    # Training/Testing structure
    train_base = datasets.ImageFolder(train_dir, transform=None)
    num_classes = len(train_base.classes)
    print('Classes:', train_base.classes)

    # If no explicit validation folder, carve out 10% from training as val
    if val_dir is None:
        n = len(train_base)
        n_val = max(1, int(0.1 * n))
        n_train = n - n_val
        torch.manual_seed(42)
        train_subset, val_subset = random_split(range(n), [n_train, n_val])
        train_ds = Subset(datasets.ImageFolder(train_dir, transform=train_tfms), train_subset.indices)
        val_ds   = Subset(datasets.ImageFolder(train_dir, transform=eval_tfms),   val_subset.indices)
    else:
        train_ds = datasets.ImageFolder(train_dir, transform=train_tfms)
        val_ds   = datasets.ImageFolder(val_dir,   transform=eval_tfms)

    test_ds  = datasets.ImageFolder(test_dir, transform=eval_tfms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f'Train/Val/Test sizes: {len(train_ds)} / {len(val_ds)} / {len(test_ds)}')

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Train/Val/Test sizes: 4569 / 571 / 572


Step 9 (Compute Class Weight)

In [ ]:
# Count samples per class from train set indices
from collections import Counter

def get_labels_from_subset(subset, root):
    # subset.dataset is ImageFolder(root), subset.indices are indices into that dataset
    targets = np.array(subset.dataset.targets)
    labels = targets[subset.indices]
    return labels

if isinstance(train_ds, Subset):
    y_train = get_labels_from_subset(train_ds, train_dir)
    counts = Counter(y_train.tolist())
    class_weights = torch.zeros(num_classes, dtype=torch.float32)
    total = sum(counts.values())
    for c in range(num_classes):
        class_weights[c] = total / (counts[c] if counts[c] > 0 else 1)
else:
    # direct ImageFolder
    counts = Counter(train_ds.targets)
    class_weights = torch.zeros(num_classes, dtype=torch.float32)
    total = sum(counts.values())
    for c in range(num_classes):
        class_weights[c] = total / (counts[c] if counts[c] > 0 else 1)

print('Class counts:', dict(counts))
print('Class weights (unnormalized):', class_weights.tolist())
class_weights = class_weights / class_weights.sum() * num_classes
print('Class weights (normalized):', class_weights.tolist())
class_weights = class_weights.to(device)

Class counts: {0: 1051, 2: 1276, 3: 1166, 1: 1076}
Class weights (unnormalized): [4.347288131713867, 4.246282577514648, 3.580720901489258, 3.918524980545044]
Class weights (normalized): [1.080553650856018, 1.0554479360580444, 0.8900171518325806, 0.9739810824394226]


Step 10 (Build and Replace)

In [ ]:
weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights)
# Replace final FC layer
model.fc = nn.Linear(model.fc.in_features, num_classes)
model = model.to(device)

# Loss, optimizer, scheduler, metrics
criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)
metric_acc = MulticlassAccuracy(num_classes=num_classes).to(device)

sum(p.numel() for p in model.parameters())/1e6, 'M parameters'

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s]


(23.516228, 'M parameters')

Step 11 (Train and Validation)

In [ ]:
from torch.cuda.amp import GradScaler, autocast
from tqdm import tqdm
import json

EPOCHS = 12
scaler = GradScaler(enabled=torch.cuda.is_available())
best_val_acc = 0.0
patience, patience_counter = 4, 0
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

def run_epoch(loader, train=True):
    if train:
        model.train()
    else:
        model.eval()

    running_loss, correct, total = 0.0, 0, 0
    loop = tqdm(loader, desc="Training" if train else "Validation", leave=False)

    for imgs, labels in loop:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad(set_to_none=True)
        with torch.set_grad_enabled(train):
            with autocast(enabled=torch.cuda.is_available()):
                logits = model(imgs)
                loss = criterion(logits, labels)
            if train:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

        running_loss += loss.item() * imgs.size(0)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        acc = correct / total
        loop.set_postfix(loss=loss.item(), acc=acc)

    return running_loss / total, correct / total


for epoch in range(1, EPOCHS + 1):
    print(f"\nEpoch {epoch:02d}/{EPOCHS}")
    tr_loss, tr_acc = run_epoch(train_loader, train=True)
    vl_loss, vl_acc = run_epoch(val_loader, train=False)
    scheduler.step()

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)

    print(f"Train Loss={tr_loss:.4f}, Acc={tr_acc:.4f} | "
          f"Val Loss={vl_loss:.4f}, Acc={vl_acc:.4f}")

    # Save best model
    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        patience_counter = 0
        best_path = os.path.join(save_dir, "resnet50_brain_tumor_best.pth")
        torch.save({
            'model': model.state_dict(),
            'classes': (train_ds.dataset.classes if isinstance(train_ds, Subset) else train_ds.classes),
            'num_classes': num_classes
        }, best_path)
        print(f"✅ Saved BEST model to {best_path}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("⏹️ Early stopping triggered.")
            break

# Save final model
final_path = os.path.join(save_dir, "resnet50_brain_tumor_final.pth")
torch.save({
    'model': model.state_dict(),
    'classes': (train_ds.dataset.classes if isinstance(train_ds, Subset) else train_ds.classes),
    'num_classes': num_classes
}, final_path)
print(f"🎉 Training done. Final model saved to {final_path}")

# Save training history
history_path = os.path.join(save_dir, "resnet50_training_history.json")
with open(history_path, "w") as f:
    json.dump(history, f, indent=4)
print(f"📝 Training history saved to {history_path}")